# 40 · Migration throughput and remaining exposure

Inputs: `holder_snapshot`, `utxo_life`, `address_key`, `address_stats`, `exchange_keys`, `custodial_keys` (02, 04, 04b, 04c, 30)

Outputs: `out40/`, zipped as `out40.zip`

Block space needed to move every exposed UTXO alive at the freeze date (2026-09-22), and the exposed balance of that cohort projected under fixed migration rates.

- `T(α)`: days needed when a share α ∈ {25%, 50%, 100%} of block space carries migration transactions. Upper bound: one transaction per UTXO. Lower bound: one transaction per key.
- `D(α) = Q-day − T(α)`: latest start date, with Q-day 5, 10 and 15 years after the freeze date.
- Fees at 1–100 sat/vB, and UTXOs worth less than the fee to spend them.

Size assumptions (vB): 1,000,000 per block, 144 blocks per day; inputs P2PKH 148, P2PK 114, P2WPKH 68, P2TR 57.5, P2SH and P2WSH calibrated in step 3; migration output 43 (P2MR/P2TR-type, BIP-360); 10.5 overhead per transaction.

Holder classes follow 30: active = last spend within 3 years of the freeze date; behaviorally unreachable = last spend (for never-spent keys, last major receipt) at least 10 years before; indeterminate otherwise. A major receipt is a UTXO of at least 1% of the key's balance, or the key's largest UTXO.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

START_EVENT      = "willow"      # starting snapshot of 30 (2024-12-08); splits new exposure in step 7
FREEZE_DATE      = "2026-09-22"  # freeze date = latest snapshot date
MIN_BTC          = 0.01          # keys with a balance below this = dust keys
SERVICE_MAX_RECV = 1000          # any address with > 1,000 lifetime receipts -> service/custodian (as in 04 and 30)
ACTIVE_YEARS     = 3             # as in 30
DORMANT_YEARS    = 10            # as in 30
MAJOR_IN_SHARE   = 0.01          # as in 30: for never-spent keys, dormancy uses the last receipt >= 1% of the balance

# block capacity
BLOCK_VB       = 1_000_000       # vB per block (4M weight units)
BLOCKS_PER_DAY = 144
ALPHAS         = [0.25, 0.50, 1.00]

# transaction size (vB)
# None = calibrated median from step 3; VB_FALLBACK below CAL_MIN_TX transactions
VB_IN = {"pubkey": 114, "pubkeyhash": 148, "witness_v0_keyhash": 68, "witness_v1_taproot": 57.5,
         "scripthash": None, "witness_v0_scripthash": None}
VB_FALLBACK = {"scripthash": 91, "witness_v0_scripthash": 104.5}   # P2SH-P2WPKH; 2-of-3 P2WSH
VB_OTHER    = 190                 # bare multisig / non-standard (only in the "all UTXOs" comparison)
OUT_VB      = 43                  # migration destination: P2MR/P2TR-type output
OVH_VB      = 10.5                # fixed overhead per transaction
CAL_DAYS    = 31                  # calibration: transactions in the 31 days before the freeze date
CAL_MIN_TX  = 1000                # if a type has fewer calibration transactions -> VB_FALLBACK

FEES     = [1, 2, 5, 10, 20, 50, 100]   # sat/vB
FEE_MAIN = 10

# remaining-exposure scenarios
Q_YEARS   = [5, 10, 15]           # Q-day this many years after the freeze date
# baseline migration rates (per year, by coin): from q2_cells of 30 (day before Willow -> freeze date, 1.79 years)
#   active (classes A+B+C of 30, non-service, L+T): y1 strict 0.175M BTC / starting 2.217M BTC / 1.79 years = 4.46%
#   exchange/custodian (L): y1 strict 0.077M / 1.595M / 1.79 years = 2.71%
#   indeterminate (D), unreachable (E), dust: no outflows at all during the observation period -> 0
R_ACT, R_SVC, R_IND = 0.0446, 0.0271, 0.0
BETA_WARN     = 0.0011            # per-warning effect: upper end of the event-study 90% interval
WARN_PER_YEAR = 4                 # scenario "a warning every quarter"
RS_BETA       = 0.0235            # Randstorm effect (21)
UB_YEARS      = 3                 # upper-bound scenario horizon

OUT_DIR = "out40"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, io, re, time, zipfile, datetime as dt
import numpy as np
import pandas as pd

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS  = f"{PROJECT_ID}.{DATASET}"
PUB = "bigquery-public-data.crypto_bitcoin"
os.makedirs(OUT_DIR, exist_ok=True)

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["utxo_life", "address_key", "address_stats", "holder_snapshot", "exchange_keys", "custodial_keys"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from earlier steps found. Spent in this Colab session: $%.2f / $%.0f" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    assert not re.search(r"CLUSTER BY[^\n]*\b(key_id|txid|spending_txid)\b", sql), f"[{name}] cannot cluster on a BYTES column"
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return True
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell (tables already built are skipped)")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))
    return True

def need(*names):
    missing = [n for n in names if not table_rows(n)]
    assert not missing, f"These tables are not built yet: {missing} (run the earlier cells first)"

def q(sql):
    return client.query(sql).to_dataframe()

def upload(name, df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    job = client.load_table_from_dataframe(
        df, f"{DS}.{name}", job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    job.result()
    print(f"[{name}] uploaded {len(df):,} rows")

def q_guarded(sql, label):
    """Queries that do not build a table are also priced first and stop above the auto-run limit."""
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{label}] estimated ${cost:.2f}, above the per-query limit")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{label}] estimated ${cost:.2f}, above the auto-run limit: set CONFIRM=True")
    print(f"[{label}] estimated {gb:,.1f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    df = job.result().to_dataframe()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": label, "job_id": job.job_id, "billed_gb": round(billed, 1), "usd": round(usd(billed), 2)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    return df

def save(df, name, **kw):
    df.to_csv(os.path.join(OUT_DIR, name), **kw)

In [ ]:
# Dates and constants
SATS_MIN = int(round(MIN_BTC * 1e8))
FREEZE = pd.Timestamp(FREEZE_DATE)
snap = q(f"""SELECT event, ANY_VALUE(snap_date) AS snap_date FROM `{DS}.holder_snapshot`
             WHERE event IN ('{START_EVENT}', 'latest') GROUP BY event""").set_index("event").snap_date
START = pd.Timestamp(snap[START_EVENT])
assert pd.Timestamp(snap["latest"]) == FREEZE, f"latest snapshot date {snap['latest']} differs from the freeze date {FREEZE_DATE}"
F  = FREEZE.strftime("%Y-%m-%d")
F1 = (FREEZE + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
CAL_START = (FREEZE - pd.Timedelta(days=CAL_DAYS - 1)).strftime("%Y-%m-%d")
YEARS_OBS = (FREEZE - START).days / 365.25
STD_TYPES = "('pubkey','pubkeyhash','witness_v0_keyhash','scripthash','witness_v0_scripthash','witness_v1_taproot')"
print(f"Freeze date {F}; start of 30 {START.date()}, observation period {YEARS_OBS:.2f} years; calibration window {CAL_START} - {F}")

## Step 1 · UTXOs at the freeze date, `q3_utxo`

All UTXOs unspent at the close of the freeze date (about 165 million), with key and snapshot group: L (legacy exposed), T (Taproot), C (hash-protected). UTXOs outside the six standard script types (bare multisig, non-standard) form group N. L and T are the migration target; C and N give the whole-network comparison.

Checks: total ≈ issued supply (about 20.09 million BTC); BTC and UTXO counts of L and T equal the latest snapshot.

In [ ]:
SQL_UTXO = f"""
CREATE OR REPLACE TABLE `{DS}.q3_utxo`
CLUSTER BY grp, type
AS
WITH lat AS (SELECT key_id, grp FROM `{DS}.holder_snapshot` WHERE event = 'latest'),
u AS (
  SELECT address, type, sats, created_date
  FROM `{DS}.utxo_life`
  WHERE created_date <= DATE '{F}'
    AND (spent_date IS NULL OR spent_date > DATE '{F}')
    AND (sats > 0 OR type IN {STD_TYPES})   -- zero-value standard outputs count (as in the snapshot); zero-value non-standard outputs such as OP_RETURN do not
)
SELECT IF(u.type IN {STD_TYPES}, k.key_id, NULL) AS key_id,
       IF(u.type IN {STD_TYPES} AND u.address IS NOT NULL, IFNULL(lat.grp, 'N'), 'N') AS grp,
       u.type, u.sats, u.created_date
FROM u
LEFT JOIN `{DS}.address_key` k ON k.address = u.address
LEFT JOIN lat ON lat.key_id = k.key_id
"""
build("q3_utxo", SQL_UTXO)

chk_utxo = q(f"""
WITH a AS (SELECT grp, COUNT(*) AS n_utxo, SUM(sats) AS sats FROM `{DS}.q3_utxo` GROUP BY grp),
     b AS (SELECT grp, SUM(n_utxo) AS n_utxo_snap, SUM(sats) AS sats_snap
           FROM `{DS}.holder_snapshot` WHERE event = 'latest' GROUP BY grp)
SELECT a.grp, a.n_utxo, b.n_utxo_snap, ROUND(a.sats/1e8, 2) AS btc, ROUND(b.sats_snap/1e8, 2) AS btc_snap
FROM a LEFT JOIN b ON b.grp = a.grp ORDER BY a.grp""")
display(chk_utxo)
tot = chk_utxo.btc.sum()
print(f"Total unspent {tot:,.2f} BTC, {chk_utxo.n_utxo.sum():,} UTXOs (supply_by_type from 01: 20,088,304 BTC)")
for g in ["L", "T"]:
    r = chk_utxo.set_index("grp").loc[g]
    assert r.n_utxo == r.n_utxo_snap and abs(r.btc - r.btc_snap) < 1e-6, f"group {g} does not match the latest snapshot"
print("Groups L and T match the latest snapshot exactly")

## Step 2 · Exposed-key attributes, `q3_key`

One row per L or T key (about 9 million): balance, UTXO count, last spend, last major receipt, exchange/custodian flag (rule of 30) and group on the day before Willow. The Willow group splits keys absent from the Willow snapshot (1.91 million BTC in the bridge table of 30) into services and individuals.

In [ ]:
need("q3_utxo")
SQL_KEY = f"""
CREATE OR REPLACE TABLE `{DS}.q3_key` AS
WITH u AS (
  SELECT key_id, ANY_VALUE(grp) AS grp, SUM(sats) AS sats, COUNT(*) AS n_utxo,
         MAX(created_date) AS newest_utxo_date, MAX(sats) AS max_utxo_sats
  FROM `{DS}.q3_utxo` WHERE grp IN ('L', 'T')
  GROUP BY key_id
),
maj AS (
  SELECT x.key_id, MAX(x.created_date) AS newest_major_date
  FROM `{DS}.q3_utxo` x JOIN u ON u.key_id = x.key_id
  -- major = a single UTXO >= 1% of the balance, or the key's largest UTXO (defined even for keys with hundreds of small UTXOs none of which reaches 1%)
  WHERE x.grp IN ('L', 'T') AND x.sats >= LEAST({MAJOR_IN_SHARE} * u.sats, u.max_utxo_sats)
  GROUP BY x.key_id
),
ka AS (
  SELECT k.key_id, MAX(s.last_spend_date) AS last_spend_date, MAX(s.n_received) AS max_addr_n_received
  FROM `{DS}.address_key` k
  JOIN u ON u.key_id = k.key_id
  JOIN `{DS}.address_stats` s ON s.address = k.address
  GROUP BY k.key_id
),
ex AS (SELECT DISTINCT key_id FROM `{DS}.exchange_keys`),
ck AS (SELECT DISTINCT key_id FROM `{DS}.custodial_keys`),
w  AS (SELECT key_id, grp AS start_grp FROM `{DS}.holder_snapshot` WHERE event = '{START_EVENT}')
SELECT u.key_id, u.grp, u.sats, u.n_utxo, u.newest_utxo_date, u.max_utxo_sats, maj.newest_major_date,
       IF(ka.last_spend_date > DATE '{F}', DATE '{F}', ka.last_spend_date) AS last_spend_date,
       ka.max_addr_n_received,
       ex.key_id IS NOT NULL AS is_exchange,
       ck.key_id IS NOT NULL AS is_custodial,
       (ex.key_id IS NOT NULL OR ck.key_id IS NOT NULL
        OR IFNULL(ka.max_addr_n_received, 0) > {SERVICE_MAX_RECV}) AS is_service,
       w.start_grp
FROM u
LEFT JOIN maj ON maj.key_id = u.key_id
LEFT JOIN ka  ON ka.key_id  = u.key_id
LEFT JOIN ex  ON ex.key_id  = u.key_id
LEFT JOIN ck  ON ck.key_id  = u.key_id
LEFT JOIN w   ON w.key_id   = u.key_id
"""
# rebuild q3_key if max_utxo_sats is missing
_old = table_rows("q3_key") and not any(f.name == "max_utxo_sats" for f in client.get_table(f"{DS}.q3_key").schema)
if _old: print("[q3_key] is an older version (some keys lack a major-receipt date); rebuilding")
build("q3_key", SQL_KEY, force=bool(_old))
chk_key = q(f"""
SELECT grp, COUNT(*) AS n_keys, COUNT(DISTINCT key_id) AS n_distinct, ROUND(SUM(sats)/1e8) AS btc,
       COUNTIF(newest_major_date IS NULL) AS no_major, COUNTIF(max_addr_n_received IS NULL) AS no_stats,
       COUNTIF(is_service) AS n_service, ROUND(SUM(IF(is_service, sats, 0))/1e8) AS service_btc
FROM `{DS}.q3_key` GROUP BY grp ORDER BY grp""")
display(chk_key)
assert (chk_key.n_keys == chk_key.n_distinct).all() and (chk_key.no_major == 0).all(), "duplicate keys or missing major-receipt dates"

## Step 3 · Input-size calibration

Non-coinbase transactions in the 31 days before the freeze date whose inputs share one script type and whose outputs are all of known types:

`vB per input = (transaction vB − Σ output vB − 10.5) / number of inputs`

The calibrated median is used for P2SH and P2WSH. For P2PKH, P2PK, P2WPKH and P2TR it is reported as a check only; observed P2TR spends include script-path and inscription transactions. The public tables carry no witness field, so sizes are backed out per transaction.

In [ ]:
tx_cols = {f.name for f in client.get_table(f"{PUB}.transactions").schema}
m0 = pd.Timestamp(CAL_START).replace(day=1).strftime("%Y-%m-%d")
month_filter = (f"AND block_timestamp_month BETWEEN DATE '{m0}' AND DATE '{F}'"
                if "block_timestamp_month" in tx_cols else "")
OUT_CASE = """CASE o.type WHEN 'pubkeyhash' THEN 34 WHEN 'witness_v0_keyhash' THEN 31 WHEN 'scripthash' THEN 32
                          WHEN 'witness_v0_scripthash' THEN 43 WHEN 'witness_v1_taproot' THEN 43
                          WHEN 'pubkey' THEN 44 END"""
SQL_CAL = f"""
WITH t AS (
  SELECT virtual_size AS vsize, input_count AS n_in,
         (SELECT COUNT(DISTINCT i.type) FROM UNNEST(inputs) i) AS n_in_types,
         (SELECT MAX(i.type) FROM UNNEST(inputs) i) AS in_type,
         (SELECT SUM({OUT_CASE}) FROM UNNEST(outputs) o) AS out_vb,
         (SELECT COUNTIF(({OUT_CASE}) IS NULL) FROM UNNEST(outputs) o) AS n_out_unknown
  FROM `{PUB}.transactions`
  WHERE block_timestamp >= TIMESTAMP('{CAL_START}') AND block_timestamp < TIMESTAMP('{F1}') {month_filter}
    AND NOT is_coinbase
),
v AS (
  SELECT in_type, IF(n_in = 1, '1', '2+') AS n_in_bin, (vsize - out_vb - {OVH_VB}) / n_in AS vb_per_in
  FROM t WHERE n_in_types = 1 AND n_out_unknown = 0 AND n_in >= 1
)
SELECT in_type, n_in_bin, COUNT(*) AS n_tx,
       APPROX_QUANTILES(vb_per_in, 100)[OFFSET(25)] AS p25,
       APPROX_QUANTILES(vb_per_in, 100)[OFFSET(50)] AS p50,
       APPROX_QUANTILES(vb_per_in, 100)[OFFSET(75)] AS p75
FROM v GROUP BY in_type, n_in_bin ORDER BY in_type, n_in_bin
"""
cal = q_guarded(SQL_CAL, "calibration")
display(cal)

# for each type, 1-input and multi-input transactions combined: median weighted by number of transactions (approximate)
cal_all = (cal.assign(w=cal.n_tx * cal.p50).groupby("in_type")
              .agg(n_tx=("n_tx", "sum"), w=("w", "sum")).assign(p50=lambda d: d.w / d.n_tx).drop(columns="w"))
VB = {}
vb_rows = []
for t, v in VB_IN.items():
    c = cal_all.p50.get(t, np.nan); n = int(cal_all.n_tx.get(t, 0))
    if v is not None:
        VB[t] = v; src = "standard"
    elif n >= CAL_MIN_TX and np.isfinite(c):
        VB[t] = round(float(c), 1); src = "calibrated median"
    else:
        VB[t] = VB_FALLBACK[t]; src = "fallback (too few transactions)"
    vb_rows.append({"type": t, "vb_used": VB[t], "source": src, "cal_p50": c, "cal_n_tx": n})
vb_tab = pd.DataFrame(vb_rows)
save(vb_tab, "q3_vb_used.csv", index=False); save(cal, "q3_calibration.csv", index=False)
display(vb_tab)
def vb_of(t): return VB.get(t, VB_OTHER)

## Step 4 · Aggregated cells (pandas from here on)

- `ucells`: UTXO count and BTC by group, service flag, dust-key flag, never-spent flag, years since last activity, Willow group and script type; per fee rate, count and BTC of uneconomic UTXOs.
- `kcells`: key counts on the same dimensions without script type, for the per-key bound. Group C has a total key count only; group N has no keys and is counted as one transaction per UTXO.

In [ ]:
need("q3_utxo", "q3_key")
VB_CASE = "CASE u.type " + " ".join(f"WHEN '{t}' THEN {v}" for t, v in VB.items()) + f" ELSE {VB_OTHER} END"
KDIM = f"""
  SELECT key_id, grp, is_service, sats < {SATS_MIN} AS key_dust, last_spend_date IS NULL AS never_spent,
         CAST(FLOOR(DATE_DIFF(DATE '{F}', COALESCE(last_spend_date, newest_major_date, newest_utxo_date), DAY) / 365.25) AS INT64) AS ref_years,
         IFNULL(start_grp, 'none') AS start_grp, sats, n_utxo
  FROM `{DS}.q3_key`"""
UNECO = ",\n".join(f"  COUNTIF(u.sats < in_vb * {f}) AS un_n_{f}, SUM(IF(u.sats < in_vb * {f}, u.sats, 0)) / 1e8 AS un_btc_{f}"
                   for f in FEES)
SQL_UCELLS = f"""
WITH k AS ({KDIM}),
u AS (SELECT *, {VB_CASE} AS in_vb FROM `{DS}.q3_utxo` u)
SELECT u.grp, IFNULL(k.is_service, FALSE) AS is_service, IFNULL(k.key_dust, FALSE) AS key_dust,
       IFNULL(k.never_spent, FALSE) AS never_spent, IFNULL(k.ref_years, -1) AS ref_years,
       IFNULL(k.start_grp, '-') AS start_grp, u.type,
       COUNT(*) AS n_utxo, SUM(u.sats) / 1e8 AS btc,
{UNECO}
FROM u LEFT JOIN k ON k.key_id = u.key_id AND u.grp IN ('L', 'T')
GROUP BY 1, 2, 3, 4, 5, 6, 7
"""
SQL_KCELLS = f"""
WITH k AS ({KDIM})
SELECT grp, is_service, key_dust, never_spent, ref_years, start_grp,
       COUNT(*) AS n_keys, SUM(sats) / 1e8 AS btc, SUM(n_utxo) AS n_utxo
FROM k GROUP BY 1, 2, 3, 4, 5, 6
UNION ALL
SELECT 'C', FALSE, FALSE, FALSE, -1, '-', COUNT(DISTINCT key_id), SUM(sats) / 1e8, COUNT(*)
FROM `{DS}.q3_utxo` WHERE grp = 'C'
"""
ucells = q_guarded(SQL_UCELLS, "ucells")
kcells = q_guarded(SQL_KCELLS, "kcells")
ATTR = ["grp", "is_service", "key_dust", "never_spent", "ref_years", "start_grp"]
for d in (ucells, kcells):
    for c in ["is_service", "key_dust", "never_spent"]: d[c] = d[c].astype(bool)
    d["ref_years"] = d.ref_years.astype(int)
# check: the L and T key dimensions of the two tables agree
a = ucells[ucells.grp.isin(["L", "T"])].groupby(ATTR)[["n_utxo", "btc"]].sum()
b = kcells[kcells.grp.isin(["L", "T"])].set_index(ATTR)[["n_utxo", "btc"]]
j = a.join(b, rsuffix="_k", how="outer").fillna(0)
assert (j.n_utxo == j.n_utxo_k).all() and np.allclose(j.btc, j.btc_k, atol=1e-6), "ucells and kcells disagree"
if "chk_utxo" not in globals():   # step 1 skipped: reference totals from the snapshot
    chk_utxo = q(f"""SELECT grp, SUM(n_utxo) AS n_utxo FROM `{DS}.holder_snapshot`
                    WHERE event = 'latest' AND grp IN ('L', 'T') GROUP BY grp""")
for g in ["L", "T"]:
    snap_r = chk_utxo.set_index("grp").loc[g]
    assert ucells[ucells.grp == g].n_utxo.sum() == snap_r.n_utxo
print(f"ucells {len(ucells):,} rows, kcells {len(kcells):,} rows, checks passed")
ucells.to_csv(os.path.join(OUT_DIR, "q3_ucells.csv.gz"), index=False, compression="gzip")
kcells.to_csv(os.path.join(OUT_DIR, "q3_kcells.csv.gz"), index=False, compression="gzip")

## Step 5 · `T(α)` and `D(α)`

Key-level classes, assigned in order as in 30: exchange/custodian → dust (balance < 0.01 BTC) → active → unreachable → indeterminate.

- Upper bound (one transaction per UTXO): `Σ UTXO × (input vB + 43 + 10.5)`
- Lower bound (one transaction per key): `Σ UTXO × input vB + keys × (43 + 10.5)`
- `T(α) = total vB / (α × 1,000,000 × 144)` days; `D(α) = Q-day − T(α)`

In [ ]:
CLS_ORDER = ["svc", "dust", "act", "ind", "dorm", "hash", "other"]
CLS_LABEL = {"svc": "Exchange/custodian", "dust": "Dust keys <0.01 BTC", "act": "Individual, confirmed active", "ind": "Individual, indeterminate",
          "dorm": "Individual, behaviorally unreachable", "hash": "Hash-protected (group C)", "other": "Bare multisig/non-standard (N)"}
CLS_FIG = {"svc": "Exchange / custodian", "dust": "Dust keys <0.01 BTC", "act": "Active holders",
          "ind": "Indeterminate", "dorm": "Behaviourally unreachable", "hash": "Hash-protected (C)", "other": "Other scripts"}

def classify(d, active=ACTIVE_YEARS, dormant=DORMANT_YEARS):
    lt = d.grp.isin(["L", "T"])
    return pd.Series(np.select(
        [d.grp == "C", ~lt, d.is_service, d.key_dust,
         ~d.never_spent & (d.ref_years < active), d.ref_years >= dormant],
        ["hash", "other", "svc", "dust", "act", "dorm"], default="ind"), index=d.index)

ucells["cls"] = classify(ucells); kcells["cls"] = classify(kcells)
ucells["in_vb"] = ucells.type.map(vb_of)
ucells["vb_in_tot"] = ucells.n_utxo * ucells.in_vb
kc_n = kcells.copy()

def demand(mask_u, mask_k, n_keys_for_other=True, uneco_fee=None):
    """Returns (UTXOs, keys, BTC, upper-bound vB, lower-bound vB)."""
    u = ucells[mask_u]; k = kc_n[mask_k]
    n_utxo = u.n_utxo.sum(); vb_in = u.vb_in_tot.sum(); btc = u.btc.sum()
    if uneco_fee is not None:           # drop UTXOs that are uneconomic at this fee rate
        un_n = u[f"un_n_{uneco_fee}"]
        n_utxo = n_utxo - un_n.sum(); vb_in = vb_in - (un_n * u.in_vb).sum(); btc = btc - u[f"un_btc_{uneco_fee}"].sum()
    n_keys = k.n_keys.sum() + (u[u.grp == "N"].n_utxo.sum() if n_keys_for_other else 0)
    n_keys = min(n_keys, n_utxo)
    per_tx = OUT_VB + OVH_VB
    return dict(n_utxo=int(n_utxo), n_keys=int(n_keys), btc=btc,
                vb_upper=vb_in + n_utxo * per_tx, vb_lower=vb_in + n_keys * per_tx)

LT_u, LT_k = ucells.grp.isin(["L", "T"]), kc_n.grp.isin(["L", "T"])
SUBSETS = [
    ("all_exposed",   "All exposed (L+T)",                         LT_u, LT_k, None),
    ("reachable",     "Reachable exposed (excl. unreachable and dust keys)",          LT_u & ucells.cls.isin(["svc", "act", "ind"]), LT_k & kc_n.cls.isin(["svc", "act", "ind"]), None),
    ("svc",           "  of which: exchange/custodian",                       ucells.cls == "svc",  kc_n.cls == "svc",  None),
    ("act",           "  of which: individual, confirmed active",                    ucells.cls == "act",  kc_n.cls == "act",  None),
    ("ind",           "  of which: individual, indeterminate",                        ucells.cls == "ind",  kc_n.cls == "ind",  None),
    ("dorm",          "Unreachable (individual, unmoved >=10 years)",                ucells.cls == "dorm", kc_n.cls == "dorm", None),
    ("dust",          "Dust keys <0.01 BTC",                          ucells.cls == "dust", kc_n.cls == "dust", None),
    ("all_exposed_econ", f"All exposed, excl. UTXOs uneconomic at {FEE_MAIN} sat/vB", LT_u, LT_k, FEE_MAIN),
    ("whole_set",     "All UTXOs network-wide (incl. hash-protected; full post-quantum migration)", ucells.grp.notna(), kc_n.grp.notna(), None),
]
rows = []
for key, lab, mu, mk, fee in SUBSETS:
    d = demand(mu, mk, uneco_fee=fee)
    for a in ALPHAS:
        cap = a * BLOCK_VB * BLOCKS_PER_DAY
        rows.append({"subset": key, "label": lab, "alpha": a, **d,
                     "days_upper": d["vb_upper"] / cap, "days_lower": d["vb_lower"] / cap})
thr = pd.DataFrame(rows)
save(thr, "q3_throughput.csv", index=False)
show = thr.pivot_table(index=["subset", "label"], columns="alpha", values=["days_lower", "days_upper"], sort=False)
show.columns = [f"{'lower' if v == 'days_lower' else 'upper'} α={a:.0%}" for v, a in show.columns]
base = thr[thr.alpha == ALPHAS[0]].set_index(["subset", "label"])[["n_utxo", "n_keys", "btc"]]
print("Days needed (lower = one transaction per key; upper = one per UTXO); columns = share alpha of block space used by migration")
display(base.join(show).style.format("{:,.0f}"))

# D(alpha): latest start date
drows = []
for h in Q_YEARS:
    qd = FREEZE + pd.DateOffset(years=h)
    for r in thr[thr.subset.isin(["all_exposed", "reachable", "whole_set"])].itertuples():
        drows.append({"q_years": h, "q_day": qd.date(), "subset": r.subset, "alpha": r.alpha,
                      "latest_start_if_merged": (qd - pd.Timedelta(days=r.days_lower)).date(),
                      "latest_start_if_per_utxo": (qd - pd.Timedelta(days=r.days_upper)).date()})
dtab = pd.DataFrame(drows)
save(dtab, "q3_deadline.csv", index=False)
print("D(alpha): latest date by which migration must start (Q-day minus the days needed)")
display(dtab.pivot_table(index=["q_years", "q_day", "subset"], columns="alpha",
                         values="latest_start_if_per_utxo", aggfunc="first", sort=False))

## Step 6 · Fees and uneconomic UTXOs

Total fees (BTC) to move all exposed coins at 1–100 sat/vB, upper and lower bounds. A UTXO is uneconomic at a given fee rate when its value is below input vB × fee rate; reported by count and BTC.

In [ ]:
frows = []
for f in FEES:
    for key in ["all_exposed", "reachable", "whole_set"]:
        r = thr[(thr.subset == key) & (thr.alpha == 1.0)].iloc[0]
        mu = dict((s[0], s[2]) for s in SUBSETS)[key]
        u = ucells[mu]
        frows.append({"fee_sat_vb": f, "subset": key,
                      "fee_btc_lower": r.vb_lower * f / 1e8, "fee_btc_upper": r.vb_upper * f / 1e8,
                      "uneconomic_utxo": int(u[f"un_n_{f}"].sum()), "uneconomic_utxo_share": u[f"un_n_{f}"].sum() / u.n_utxo.sum(),
                      "uneconomic_btc": u[f"un_btc_{f}"].sum()})
fee = pd.DataFrame(frows)
save(fee, "q3_fees.csv", index=False)
display(fee[fee.subset == "all_exposed"].style.format({"fee_btc_lower": "{:,.1f}", "fee_btc_upper": "{:,.1f}",
        "uneconomic_utxo": "{:,.0f}", "uneconomic_utxo_share": "{:.1%}", "uneconomic_btc": "{:,.2f}"}).hide(axis="index"))

# where the uneconomic UTXOs are (main fee rate)
unc = (ucells[LT_u].groupby(["cls", "type"])[["n_utxo", f"un_n_{FEE_MAIN}", "btc", f"un_btc_{FEE_MAIN}"]].sum()
       .query("n_utxo > 0"))
unc["share_uneconomic"] = unc[f"un_n_{FEE_MAIN}"] / unc.n_utxo
save(unc.reset_index(), "q3_uneconomic_by_cls_type.csv", index=False)
print(f"UTXOs uneconomic at {FEE_MAIN} sat/vB, by class x script type")
display(unc.style.format("{:,.0f}", subset=["n_utxo", f"un_n_{FEE_MAIN}"])
           .format("{:,.2f}", subset=["btc", f"un_btc_{FEE_MAIN}"]).format("{:.1%}", subset=["share_uneconomic"]))

## Step 7 · Composition of the exposed stock

Exposed L+T stock at the freeze date by class (BTC, keys, UTXOs). The two new-exposure rows of the bridge table in 30 (keys absent from the Willow snapshot, about 1.91 million BTC; keys in group C at Willow and exposed later, about 0.18 million BTC) are split into services and individuals.

In [ ]:
stock = (kcells[kcells.grp.isin(["L", "T"])].groupby(["grp", "cls"])[["n_keys", "n_utxo", "btc"]].sum()
         .reindex(pd.MultiIndex.from_product([["L", "T"], CLS_ORDER[:5]]), fill_value=0))
stock["share_btc"] = stock.btc / stock.groupby(level=0).btc.transform("sum")
save(stock.reset_index().rename(columns={"level_0": "grp", "level_1": "cls"}), "q3_stock_by_cls.csv", index=False)
print("Composition of the exposed stock at the freeze date")
display(stock.style.format({"n_keys": "{:,.0f}", "n_utxo": "{:,.0f}", "btc": "{:,.0f}", "share_btc": "{:.1%}"}))

ne = (kcells[kcells.grp == "L"].assign(src=lambda d: d.start_grp.map(
        {"L": "Already group L at Willow", "T": "Group T at Willow", "C": "Group C at Willow (exposed later)", "none": "Not in Willow snapshot"}))
      .groupby(["src", "is_service"])[["n_keys", "btc"]].sum().unstack("is_service", fill_value=0))
ne.columns = [f"{a}_{'service' if b else 'individual'}" for a, b in ne.columns]
save(ne.reset_index(), "q3_new_exposure_split.csv", index=False)
print("Group-L stock at the freeze date by group at Willow x service/individual (compare the bridge of 30: not in snapshot 1.913M, group C 0.182M)")
display(ne.style.format("{:,.0f}"))
# annual inflow of newly exposed individual keys (not L/T at Willow; excludes new receipts by existing keys)
new_ind = kcells[(kcells.grp.isin(["L", "T"])) & kcells.start_grp.isin(["C", "none"]) & ~kcells.is_service
                 & kcells.cls.isin(["act", "ind"])].btc.sum()
INFLOW_PER_YEAR = new_ind / YEARS_OBS
print(f"Newly exposed individual keys (active + indeterminate): balance at the freeze date {new_ind:,.0f} BTC -> about {INFLOW_PER_YEAR:,.0f} BTC per year")

## Step 8 · Remaining-exposure projections (monthly, 15 years)

Closed cohort: the L+T stock exposed at the freeze date, decaying monthly by class.

| Scenario | Active | Exchange/custodian | Indeterminate | Unreachable, dust |
|---|---|---|---|---|
| Status quo | 4.46% per year (from 30) | 2.71% per year | 0 | 0 |
| Quarterly warnings | status quo + 0.11% per warning (upper end of the event-study 90% interval) | status quo | 0 | 0 |
| Annual personal alert | status quo + 2.35% once a year (Randstorm effect, 21) | status quo | 0 | 0 |
| Upper bound | linear, fully moved within 3 years | within 3 years | within 3 years | 0 |

Assumptions: key-share effects are applied as coin shares; new exposure is not projected (step 7 reports it).

In [ ]:
S0 = kcells[kcells.grp.isin(["L", "T"])].groupby("cls").btc.sum().reindex(CLS_ORDER[:5], fill_value=0).to_dict()
RATE = {"act": R_ACT, "svc": R_SVC, "ind": R_IND, "dorm": 0.0, "dust": 0.0}
SCEN = {"status_quo": "Status quo", "quarterly_warning": "Quarterly warnings (event-study upper bound)",
        "personal_alert": "Annual personal alert (Randstorm)",
        "upper_bound": f"Upper bound: all coins except unreachable move within {UB_YEARS} years"}
SCEN_FIG = {"status_quo": "Status quo", "quarterly_warning": "Quarterly warnings (event-study upper bound)",
           "personal_alert": "Annual personal alert (Randstorm-like)",
           "upper_bound": f"Upper bound: all reachable migrate in {UB_YEARS}y"}
H = max(Q_YEARS) * 12
recs = []
for sc in SCEN:
    S = dict(S0)
    for m in range(0, H + 1):
        if m > 0:
            if sc == "upper_bound":
                for c in ["act", "svc", "ind"]:
                    S[c] = S0[c] * max(0.0, 1 - m / (12 * UB_YEARS))
            else:
                for c in S: S[c] *= (1 - RATE[c] / 12)
                if sc == "quarterly_warning" and m % (12 // WARN_PER_YEAR) == 0: S["act"] *= (1 - BETA_WARN)
                if sc == "personal_alert" and m % 12 == 1: S["act"] *= (1 - RS_BETA)
        recs.append({"scenario": sc, "month": m, **S, "total": sum(S.values())})
curve = pd.DataFrame(recs)
save(curve, "q3_remaining_curve.csv", index=False)
T0 = sum(S0.values())
rem = curve[curve.month.isin([12 * h for h in Q_YEARS])].assign(q_years=lambda d: d.month // 12)
rem["share_of_freeze"] = rem.total / T0
rem["dormant_floor_share"] = (rem.dorm + rem.dust) / rem.total
save(rem, "q3_remaining_at_qday.csv", index=False)
print(f"Exposed stock at the freeze date {T0/1e6:.2f}M BTC; unreachable + dust {(S0['dorm']+S0['dust'])/1e6:.2f}M (does not fall in any voluntary scenario)")
display(rem.pivot_table(index="scenario", columns="q_years", values="total", sort=False).reindex(list(SCEN))
           .rename(index=SCEN).div(1e6).style.format("{:.2f}M"))
display(rem.pivot_table(index="scenario", columns="q_years", values="share_of_freeze", sort=False).reindex(list(SCEN))
           .rename(index=SCEN).style.format("{:.0%}"))

## Step 9 · Sensitivity

Activity window 1/3/5 years × dormancy threshold 5/10/15 years: `T(25%)` for reachable exposed coins and the status-quo remainder after 10 years. Output size 31 vB instead of 43.

In [ ]:
srows = []
for a_y in [1, 3, 5]:
    for d_y in [5, 10, 15]:
        uc = ucells.assign(cls=classify(ucells, a_y, d_y)); kc = kcells.assign(cls=classify(kcells, a_y, d_y))
        mu = uc.grp.isin(["L", "T"]) & uc.cls.isin(["svc", "act", "ind"])
        mk = kc.grp.isin(["L", "T"]) & kc.cls.isin(["svc", "act", "ind"])
        u, k = uc[mu], kc[mk]
        vb_up = (u.n_utxo * (u.in_vb + OUT_VB + OVH_VB)).sum()
        vb_lo = u.vb_in_tot.sum() + k.n_keys.sum() * (OUT_VB + OVH_VB)
        s = kc[kc.grp.isin(["L", "T"])].groupby("cls").btc.sum()
        r10 = sum(s.get(c, 0) * (1 - RATE[c] / 12) ** 120 for c in ["act", "svc", "ind", "dorm", "dust"])
        srows.append({"active_years": a_y, "dormant_years": d_y, "reachable_btc": u.btc.sum(),
                      "dorm_btc": s.get("dorm", 0), "act_btc": s.get("act", 0),
                      "days25_lower": vb_lo / (0.25 * BLOCK_VB * BLOCKS_PER_DAY),
                      "days25_upper": vb_up / (0.25 * BLOCK_VB * BLOCKS_PER_DAY),
                      "remaining_10y_status_quo": r10})
sens = pd.DataFrame(srows)
save(sens, "q3_sensitivity.csv", index=False)
display(sens.style.format("{:,.0f}", subset=["reachable_btc", "dorm_btc", "act_btc", "remaining_10y_status_quo"])
            .format("{:,.1f}", subset=["days25_lower", "days25_upper"]).hide(axis="index"))
u = ucells[LT_u]
for ov in [OUT_VB, 31]:
    vb_up = (u.n_utxo * (u.in_vb + ov + OVH_VB)).sum()
    print(f"Output {ov} vB: all exposed, T(25%) upper bound {vb_up / (0.25 * BLOCK_VB * BLOCKS_PER_DAY):,.1f} days")

## Step 10 · Figures

- `fig_40_throughput.png`: days needed per segment (log scale); one bar per α = 25/50/100%, bar ends at the per-key and per-UTXO bounds.
- `fig_40_remaining.png`: remaining exposed BTC by scenario; dashed line = unreachable + dust.

In [ ]:
import matplotlib.pyplot as plt
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
ACOL = {0.25: "#2a78d6", 0.50: "#eb6834", 1.00: "#1baf7a"}
FIG_SUB = ["all_exposed", "reachable", "svc", "act", "ind", "dorm", "dust", "whole_set"]
FIG_LABEL = {"all_exposed": "All exposed (L+T)", "reachable": "Reachable exposed", "svc": "  Exchange / custodian",
          "act": "  Active holders", "ind": "  Indeterminate", "dorm": "Behaviourally unreachable",
          "dust": "Dust keys <0.01 BTC", "whole_set": "Entire UTXO set (full PQ migration)"}

fig, ax = plt.subplots(figsize=(10, 4.2))
for yi, s in enumerate(FIG_SUB):
    for j, a in enumerate(ALPHAS):
        r = thr[(thr.subset == s) & (thr.alpha == a)].iloc[0]
        y = yi + (j - 1) * 0.22
        ax.plot([r.days_lower, r.days_upper], [y, y], color=ACOL[a], lw=3, solid_capstyle="round")
    r = thr[(thr.subset == s) & (thr.alpha == 0.25)].iloc[0]
    dtxt = f"{r.days_upper:,.0f}" if r.days_upper >= 10 else f"{r.days_upper:.1f}"
    ax.text(r.days_upper * 1.12, yi - 0.22, f"{dtxt} d at α=25% · {r.n_utxo/1e6:,.1f}M UTXO · {r.btc/1e6:,.2f}M BTC",
            va="center", fontsize=7.5, color=INK2)
ax.set_yticks(range(len(FIG_SUB)), [FIG_LABEL[s] for s in FIG_SUB], fontsize=8.5, color=INK)
ax.invert_yaxis(); ax.set_xscale("log")
ax.set_xlabel("days of block space needed (lower = one tx per key; upper = one tx per UTXO)", color=INK2)
ax.grid(axis="x", color=GRID, lw=0.8); [ax.spines[sp].set_visible(False) for sp in ["top", "right"]]
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0], [0], color=ACOL[a], lw=3, label=f"α = {a:.0%} of block space") for a in ALPHAS],
          frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3)
ax.set_title(f"How long would migration take? UTXO set on {F}", fontsize=10, loc="left", color=INK)
plt.savefig(os.path.join(OUT_DIR, "fig_40_throughput.png"), dpi=150, bbox_inches="tight"); plt.show()

SCOL = {"status_quo": "#0b0b0b", "quarterly_warning": "#2a78d6", "personal_alert": "#eb6834",
        "status_quo_inflow": "#8a8983", "upper_bound": "#1baf7a"}
fig, ax = plt.subplots(figsize=(9, 4))
yrs = curve.month.unique() / 12
for sc in SCEN:
    c = curve[curve.scenario == sc]
    ax.plot(c.month / 12, c.total / 1e6, color=SCOL[sc], lw=2, label=SCEN_FIG[sc])
    if sc in ("status_quo", "personal_alert", "upper_bound"):
        end = c.iloc[-1]; ax.text(15.2, end.total / 1e6, f"{end.total/1e6:.2f}M", fontsize=8, va="center", color=SCOL[sc])
floor = (S0["dorm"] + S0["dust"]) / 1e6
ax.axhline(floor, color="#e87ba4", lw=1.5, ls=":")
ax.text(0.1, floor * 0.93, f"unreachable + dust floor {floor:.2f}M", color="#c0507e", fontsize=8, va="top")
for h in Q_YEARS: ax.axvline(h, color=GRID, lw=1, zorder=0)
ax.set_xlim(0, 15); ax.set_ylim(0, None)
ax.set_xlabel(f"years after {F} (Q-day at 5 / 10 / 15 years)", color=INK2); ax.set_ylabel("exposed BTC (millions)", color=INK2)
[ax.spines[sp].set_visible(False) for sp in ["top", "right"]]
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
ax.set_title(f"Exposed supply of {F} remaining under voluntary migration (closed cohort)", fontsize=10, loc="left", color=INK)
plt.savefig(os.path.join(OUT_DIR, "fig_40_remaining.png"), dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
for rec in JOBS[-4:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
for t in ["q3_utxo", "q3_key"]:
    n = table_rows(t)
    print(f"{t:10s}", f"{n:,} rows" if n else "not built yet")

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "config": {"freeze": F, "start_event": START_EVENT, "min_btc": MIN_BTC, "service_max_recv": SERVICE_MAX_RECV,
                     "active_years": ACTIVE_YEARS, "dormant_years": DORMANT_YEARS, "block_vb": BLOCK_VB,
                     "blocks_per_day": BLOCKS_PER_DAY, "alphas": ALPHAS, "vb_used": VB, "vb_other": VB_OTHER,
                     "out_vb": OUT_VB, "ovh_vb": OVH_VB, "cal_window": [CAL_START, F], "fees": FEES,
                     "rates": {"act": R_ACT, "svc": R_SVC, "ind": R_IND}, "beta_warn": BETA_WARN,
                     "warn_per_year": WARN_PER_YEAR, "rs_beta": RS_BETA, "ub_years": UB_YEARS,
                     "inflow_per_year": INFLOW_PER_YEAR, "years_obs": YEARS_OBS},
          "ledger": LEDGER,
          "tables": [table_info(t) for t in ["q3_utxo", "q3_key"] if table_rows(t)]}
for k, v in [("check_utxo", "chk_utxo"), ("check_key", "chk_key"), ("vb", "vb_tab"), ("throughput", "thr"),
             ("fees_all_exposed", "fee[fee.subset == 'all_exposed']"), ("stock", "stock.reset_index()"),
             ("remaining", "rem"), ("sensitivity", "sens")]:
    try: report[k] = eval(v).astype(str).to_dict(orient="records")
    except NameError: pass
with open(os.path.join(OUT_DIR, "build_report_40.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
with zipfile.ZipFile("out40.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, fn), arcname=f"out40/{fn}")
print("Packed:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download("out40.zip")
except Exception:
    print("out40.zip is in the working directory")